In [258]:
import pandas as pd

In [259]:
df = pd.read_csv("sampled_genetic_disorders.csv")
df.shape

(5307, 57)

In [260]:
df = df.replace(r'\n+|\r+', ' ', regex=True)

In [261]:
sampled_df = df.dropna(axis=0, how='any')
sampled_df.shape

(5307, 57)

In [262]:
#sampled_df.head()

In [263]:
sampled_df.to_csv("sampled_genetic_disorders.csv")

In [264]:
sampled_df.loc[:, "Severity of Symptoms"] = sampled_df.loc[:, ['Symptom 1', 'Symptom 2', 'Symptom 3', 'Symptom 4', 'Symptom 5']].sum(axis=1)
sampled_df.loc[:, "Severity of Symptoms"]

0       4.0
1       3.0
2       3.0
3       2.0
4       1.0
       ... 
5302    4.0
5303    1.0
5304    3.0
5305    3.0
5306    2.0
Name: Severity of Symptoms, Length: 5307, dtype: float64

In [265]:
sampled_df.to_csv("sampled_genetic_disorders.csv")

In [266]:
df.columns

Index(['Unnamed: 0.10', 'Unnamed: 0.9', 'Unnamed: 0.8', 'Unnamed: 0.7',
       'Unnamed: 0.6', 'Unnamed: 0.5', 'Unnamed: 0.4', 'Unnamed: 0.3',
       'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'Patient Id',
       'Patient Age', 'Genes in mother's side', 'Inherited from father',
       'Maternal gene', 'Paternal gene', 'Blood cell count (mcL)',
       'Patient First Name', 'Family Name', 'Father's name', 'Mother's age',
       'Father's age', 'Institute Name', 'Location of Institute', 'Status',
       'Respiratory Rate (breaths/min)', 'Heart Rate (rates/min', 'Test 1',
       'Test 2', 'Test 3', 'Test 4', 'Test 5', 'Parental consent', 'Follow-up',
       'Gender', 'Birth asphyxia',
       'Autopsy shows birth defect (if applicable)', 'Place of birth',
       'Folic acid details (peri-conceptional)',
       'H/O serious maternal illness', 'H/O radiation exposure (x-ray)',
       'H/O substance abuse', 'Assisted conception IVF/ART',
       'History of anomalies in previous pregnancies

In [267]:
df.rename(columns={"Genes in mother's side": "Inherited from mother"}, inplace=True)

In [269]:
df_sankey = df[['Maternal gene', 'Paternal gene', 'Inherited from father', 'Inherited from mother', 'Genetic Disorder', 'Disorder Subclass']]
df_sankey

,Maternal gene,Paternal gene,Inherited from father,Inherited from mother,Genetic Disorder,Disorder Subclass
0,Yes,No,No,No,Mitochondrial genetic inheritance disorders,Leigh syndrome
1,Yes,Yes,Yes,No,Multifactorial genetic inheritance disorders,Diabetes
2,Yes,No,No,Yes,Mitochondrial genetic inheritance disorders,Leigh syndrome
3,Yes,No,Yes,Yes,Mitochondrial genetic inheritance disorders,Mitochondrial myopathy
4,Yes,Yes,Yes,No,Single-gene inheritance diseases,Hemochromatosis
...,...,...,...,...,...,...
5302,No,No,Yes,Yes,Single-gene inheritance diseases,Cystic fibrosis
5303,No,Yes,Yes,Yes,Mitochondrial genetic inheritance disorders,Mitochondrial myopathy
5304,No,Yes,Yes,Yes,Single-gene inheritance diseases,Cystic fibrosis
5305,Yes,No,No,Yes,Mitochondrial genetic inheritance disorders,Leigh syndrome


In [270]:
df_sankey.groupby('Inherited from father')['Genetic Disorder'].count()

Inherited from father
No     3180
Yes    2127
Name: Genetic Disorder, dtype: int64

In [271]:
df_sankey['Genetic Disorder'].value_counts()

Genetic Disorder
Mitochondrial genetic inheritance disorders     2720
Single-gene inheritance diseases                2024
Multifactorial genetic inheritance disorders     563
Name: count, dtype: int64

In [272]:
df_sankey['Disorder Subclass'].value_counts()

Disorder Subclass
Leigh syndrome                         1347
Mitochondrial myopathy                 1201
Cystic fibrosis                         935
Tay-Sachs                               741
Diabetes                                499
Hemochromatosis                         348
Leber's hereditary optic neuropathy     172
Alzheimer's                              39
Cancer                                   25
Name: count, dtype: int64

In [273]:
df_sankey.groupby('Genetic Disorder')['Disorder Subclass'].count()

Genetic Disorder
Mitochondrial genetic inheritance disorders     2720
Multifactorial genetic inheritance disorders     563
Single-gene inheritance diseases                2024
Name: Disorder Subclass, dtype: int64

In [274]:
def get_origin(row):
    mom = row['Inherited from mother'] == 'Yes'
    dad = row['Inherited from father'] == 'Yes'
    
    if mom and dad: return 'Both Parents'
    if mom:         return 'Mother Only'
    if dad:         return 'Father Only'
    return 'Neither Parent'

In [275]:
def get_ancestral_origin(row):
    mom = row['Maternal gene'] == 'Yes'
    dad = row['Paternal gene'] == 'Yes'
    
    if mom and dad: return 'Both Ancestral Sides'
    if mom:         return 'Maternal Only'
    if dad:         return 'Paternal Only'
    return 'Neither Ancestral'

In [276]:
df_sankey['Ancestral Origin'] = df_sankey.apply(get_ancestral_origin, axis=1)

In [277]:
df_sankey['Origin'] = df_sankey.apply(get_origin, axis=1)

In [280]:
level_0 = df_sankey.groupby(['Ancestral Origin', 'Origin']).size().reset_index(name='value')
level_0.columns = ['source', 'target', 'value']
level_0['a_origin_id'] = level_0.source

In [288]:
level_1 = df_sankey.groupby(['Ancestral Origin', 'Origin', 'Genetic Disorder']).size().reset_index(name='value')
level_1.columns = ['a_origin_id', 'source', 'target', 'value']
level_1['origin_id'] = level_1.source


In [289]:
level_2 = df_sankey.groupby(['Ancestral Origin', 'Origin', 'Genetic Disorder', 'Disorder Subclass']).size().reset_index(name='value')
level_2.columns = ['a_origin_id', 'origin_id', 'source', 'target', 'value']
level_2.head()

,a_origin_id,origin_id,source,target,value
0,Both Ancestral Sides,Both Parents,Mitochondrial genetic inheritance disorders,Leber's hereditary optic neuropathy,30
1,Both Ancestral Sides,Both Parents,Mitochondrial genetic inheritance disorders,Leigh syndrome,81
2,Both Ancestral Sides,Both Parents,Mitochondrial genetic inheritance disorders,Mitochondrial myopathy,36
3,Both Ancestral Sides,Both Parents,Multifactorial genetic inheritance disorders,Alzheimer's,13
4,Both Ancestral Sides,Both Parents,Multifactorial genetic inheritance disorders,Diabetes,69


In [290]:
links = pd.concat([level_0, level_1, level_2], axis=0)

In [291]:
links

,source,target,value,a_origin_id,origin_id
0,Both Ancestral Sides,Both Parents,358,Both Ancestral Sides,NaN
1,Both Ancestral Sides,Father Only,208,Both Ancestral Sides,NaN
2,Both Ancestral Sides,Mother Only,450,Both Ancestral Sides,NaN
3,Both Ancestral Sides,Neither Parent,282,Both Ancestral Sides,NaN
4,Maternal Only,Both Parents,422,Maternal Only,NaN
...,...,...,...,...,...
122,Multifactorial genetic inheritance disorders,Cancer,2,Paternal Only,Neither Parent
123,Multifactorial genetic inheritance disorders,Diabetes,9,Paternal Only,Neither Parent
124,Single-gene inheritance diseases,Cystic fibrosis,27,Paternal Only,Neither Parent
125,Single-gene inheritance diseases,Hemochromatosis,33,Paternal Only,Neither Parent


In [294]:
links.to_csv('links.csv')

In [292]:
unique_nodes = list(pd.unique(links[['source', 'target']].values.ravel('K')))
mapping = {name: i for i, name in enumerate(unique_nodes)}

links['source_idx'] = links['source'].map(mapping)
links['target_idx'] = links['target'].map(mapping)

In [ ]:
color_map = {
    'Mother Only': 'rgba(180, 50, 50, 0.4)', # Red
    'Father Only': 'rgba(50, 112, 168, 0.4)', # Blue
    'Both Parents': 'rgba(142, 68, 173, 0.4)', # Purple
    'Neither Parent': 'rgba(230, 126, 34, 0.4)' # Orange
}

links['link_color'] = links['origin_id'].map(color_map)

node_colors = [color_map[node] if node in color_map else "#d3d3d3" for node in unique_nodes]

In [ ]:
fig = go.Figure(data=[go.Sankey(
    node = dict(
        label = unique_nodes, 
        color = node_colors,
        pad = 20, thickness = 15
    ),
    link = dict(
        source = links['source_idx'],
        target = links['target_idx'],
        value = links['value'],
        color = links['link_color']
    )
)])

fig.update_layout(title_text="Genetic Inheritance: Color-Coded Flow", font_size=12)
fig.show()